# Fine-Tuning Indic-Speak on Marathi with LoRA

This notebook runs the complete Marathi TTS fine-tuning and inference pipeline on a Kaggle NVIDIA T4 GPU.
- **Base Model**: `bodhan-ai/indic-speak` (3.3B parameters, SNAC 24kHz codec)
- **Dataset**: `snorbyte/indic-tts-sample-snac-encoded` (filtered Marathi subset)
- **Precision**: `fp16` (matched to Kaggle T4 GPU)

In [ ]:
# 1. Clone repository and install dependencies
!git clone https://github.com/mehersoni/indic-speak-marathi-finetune.git
%cd indic-speak-marathi-finetune

# Note: torch is omitted to prevent breaking Kaggle pre-installed GPU match
!pip install -q "transformers>=5" peft accelerate snac soundfile datasets pandas pyarrow huggingface_hub pyyaml

In [ ]:
# 2. Hugging Face Authentication
import os
from huggingface_hub import login

# Set your Hugging Face token (with access to bodhan-ai/indic-speak and snorbyte dataset)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face authenticated successfully.")
else:
    print("Please set HF_TOKEN in Kaggle Secrets or environment.")

In [ ]:
# 3. Prepare and filter Marathi dataset (Threshold: 1400 tokens)
!python scripts/prepare_dataset.py

In [ ]:
# 4. Run GPU Smoke Test (Forward, Backward, Optimizer, NaN/Inf gradient checks)
!python scripts/smoke_test.py --config configs/marathi_lora.yaml

In [ ]:
# 5. Train LoRA adapter on Marathi speech dataset
!python src/train.py --config configs/marathi_lora.yaml

In [ ]:
# 6. Generate speech with fine-tuned LoRA adapter
!python src/inference.py \
    --model bodhan-ai/indic-speak \
    --adapter outputs/lora_marathi/final_adapter \
    --out-dir outputs/examples

In [ ]:
# 7. Audio Playback
import IPython.display as ipd
from pathlib import Path

audio_files = sorted(Path("outputs/examples").glob("*.wav"))
print(f"Found {len(audio_files)} generated audio files:")
for f in audio_files:
    print(f"\n--- {f.name} ---")
    ipd.display(ipd.Audio(str(f)))